# Case Study 1: Random Fourier Boundaries

This notebook creates the nine random Fourier boundaries used for Case Study 1 and places non-overlapping particles inside each mapped domain. Shape numbering starts from 1.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from scipy.spatial import cKDTree
except Exception:
    cKDTree = None

TWO_PI = 2.0 * np.pi

FIGURE_DIR = Path("Figures")
VIDEO_DIR = Path("Videos")
FIGURE_DIR.mkdir(exist_ok=True)
VIDEO_DIR.mkdir(exist_ok=True)


In [ ]:
def fourier_curve(theta, a, b, c, d):
    """
    X(theta) = a0 + sum_k (a_k cos(k*theta) + b_k sin(k*theta))
    Y(theta) = c0 + sum_k (c_k cos(k*theta) + d_k sin(k*theta))
    """
    X = a[0] * np.ones_like(theta)
    Y = c[0] * np.ones_like(theta)
    K = len(a) - 1
    for k in range(1, K + 1):
        X += a[k] * np.cos(k * theta) + b[k] * np.sin(k * theta)
        Y += c[k] * np.cos(k * theta) + d[k] * np.sin(k * theta)
    return X, Y


def fourier_curve_derivative(theta, a, b, c, d):
    """Derivative of the Fourier boundary curve."""
    Xp = np.zeros_like(theta)
    Yp = np.zeros_like(theta)
    K = len(a) - 1
    for k in range(1, K + 1):
        Xp += -k * a[k] * np.sin(k * theta) + k * b[k] * np.cos(k * theta)
        Yp += -k * c[k] * np.sin(k * theta) + k * d[k] * np.cos(k * theta)
    return Xp, Yp


def area_weight(theta, a, b, c, d):
    """
    For x = r f(theta), y = r g(theta), the theta part of |det J| is
    |f g' - g f'|.
    """
    f, g = fourier_curve(theta, a, b, c, d)
    fp, gp = fourier_curve_derivative(theta, a, b, c, d)
    return np.abs(f * gp - g * fp)


def make_theta_sampler(a, b, c, d, n=10000):
    theta = np.linspace(0.0, TWO_PI, n, endpoint=False)
    w = np.maximum(area_weight(theta, a, b, c, d), 0.0)
    cdf = np.cumsum(w)
    cdf = cdf / cdf[-1]
    return theta, cdf


def sample_theta(rng, theta_table, cdf_table):
    return np.interp(rng.random(), cdf_table, theta_table)


def map_particles(q, a, b, c, d):
    """Map q = (r, theta) to physical x = (x, y)."""
    r = q[:, 0]
    theta = q[:, 1]
    f, g = fourier_curve(theta, a, b, c, d)
    x = np.empty((len(q), 2))
    x[:, 0] = r * f
    x[:, 1] = r * g
    return x

In [ ]:
def minimum_neighbour_distance(x):
    if len(x) < 2:
        return np.inf, np.full(len(x), np.inf)

    if cKDTree is not None:
        tree = cKDTree(x)
        dist, _ = tree.query(x, k=2)
        return dist[:, 1].min(), dist[:, 1]

    d_min = np.inf
    nearest = np.full(len(x), np.inf)
    for i in range(len(x)):
        d2 = np.sum((x - x[i])**2, axis=1)
        d2[i] = np.inf
        nearest[i] = np.sqrt(np.min(d2))
        d_min = min(d_min, nearest[i])
    return d_min, nearest


def generate_particles_no_overlap(
    N,
    a, b, c, d,
    particle_radius,
    r_min=0.2,
    r_max=0.9,
    safety=1.05,
    max_trials=200000,
    seed=1,
):
    """
    Place particles in physical space using the curvilinear map.

    Candidates are sampled in q-space and accepted only if their physical
    distance from already accepted particles is larger than
    safety * 2 * particle_radius.
    """
    rng = np.random.default_rng(seed)

    min_dist = safety * 2.0 * particle_radius
    min_dist2 = min_dist**2

    theta_table, cdf_table = make_theta_sampler(a, b, c, d)

    q_acc = np.empty((N, 2))
    x_acc = np.empty((N, 2))

    count = 0
    trials = 0

    while count < N and trials < max_trials:
        trials += 1

        u = rng.random()
        r = np.sqrt(r_min**2 + u * (r_max**2 - r_min**2))
        theta = sample_theta(rng, theta_table, cdf_table)

        q_new = np.array([[r, theta]])
        x_new = map_particles(q_new, a, b, c, d)[0]

        if count == 0:
            accept = True
        else:
            d2 = np.sum((x_acc[:count] - x_new)**2, axis=1)
            accept = np.min(d2) > min_dist2

        if accept:
            q_acc[count] = q_new[0]
            x_acc[count] = x_new
            count += 1

    q_acc = q_acc[:count]
    x_acc = x_acc[:count]

    if count < N:
        warnings.warn(
            f"Only {count} particles were placed out of requested N={N}. "
            "Try reducing particle_radius or safety, increasing r_max, "
            "decreasing r_min, or increasing max_trials."
        )

    d_min, nearest = minimum_neighbour_distance(x_acc)

    info = {
        "requested": N,
        "placed": count,
        "trials": trials,
        "particle_diameter": 2.0 * particle_radius,
        "minimum_allowed_distance": min_dist,
        "minimum_actual_distance": d_min,
    }

    return q_acc, x_acc, nearest, info

In [ ]:
def make_random_fourier_shape(
    K=8,
    seed=None,
    amp=0.25,
    decay=1.0,
    min_radius=0.35,
    min_jacobian=0.03,
    max_tries=5000,
):
    """
    Random Fourier boundary for the radial map x = r f(theta), y = r g(theta).

    The rejection tests keep only shapes with a usable radial map.
    """
    rng = np.random.default_rng(seed)
    theta_test = np.linspace(0.0, TWO_PI, 4000, endpoint=False)

    for _ in range(max_tries):
        aa = np.zeros(K + 1)
        bb = np.zeros(K + 1)
        cc = np.zeros(K + 1)
        dd = np.zeros(K + 1)

        aa[1] = 1.0
        dd[1] = 1.0

        for k in range(2, K + 1):
            scale = amp / (k ** decay)
            aa[k] = scale * rng.normal()
            bb[k] = scale * rng.normal()
            cc[k] = scale * rng.normal()
            dd[k] = scale * rng.normal()

        Xs, Ys = fourier_curve(theta_test, aa, bb, cc, dd)
        Xp, Yp = fourier_curve_derivative(theta_test, aa, bb, cc, dd)

        radius = np.sqrt(Xs**2 + Ys**2)
        jac_theta = Xs * Yp - Ys * Xp

        if np.min(radius) < min_radius:
            continue
        if np.min(jac_theta) < min_jacobian:
            continue

        return aa, bb, cc, dd

    raise RuntimeError("Could not generate a valid random shape.")

## Parameters

In [ ]:
# Shape generation parameters, matching the source notebook.
n_shapes = 9
shape_K = 6
shape_amp = 0.22
shape_min_radius = 0.45
shape_min_jacobian = 0.15
shape_seed_base = 100

# Particle placement parameters, matching the source notebook.
particle_radius = 0.015
N = 1200
r_min = 0.05
r_max = 0.95
safety = 1.05
max_trials = 200000
particle_seed_base = 2

## Create The Nine Shapes

In [ ]:
random_shapes = []
shape_curves = []

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.ravel()

theta_plot = np.linspace(0.0, TWO_PI, 1000)

for shape_id in range(n_shapes):
    shape_number = shape_id + 1
    aa, bb, cc, dd = make_random_fourier_shape(
        K=shape_K,
        seed=shape_seed_base + shape_id,
        amp=shape_amp,
        min_radius=shape_min_radius,
        min_jacobian=shape_min_jacobian,
    )

    random_shapes.append((aa, bb, cc, dd))

    Xs, Ys = fourier_curve(theta_plot, aa, bb, cc, dd)
    shape_curves.append((Xs, Ys))

    ax = axes[shape_id]
    ax.plot(Xs, Ys, color="black", lw=1.0)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(f"shape {shape_number}")
    ax.grid(True)

plt.tight_layout()
fig.savefig(FIGURE_DIR / "case_1_random_shapes.png", dpi=300, bbox_inches="tight")
plt.show()


## Particle Distribution In Physical And Curvilinear Space

In [ ]:
ncols = 3
nrows = int(np.ceil(n_shapes / ncols))

all_results = []

fig = plt.figure(figsize=(12, 4 * nrows))
outer = fig.add_gridspec(nrows, ncols, wspace=0.25, hspace=0.35)

for shape_id in range(n_shapes):
    shape_number = shape_id + 1
    a, b, c, d = random_shapes[shape_id]

    X, Y = shape_curves[shape_id]

    q_particles, x_particles, nearest, info = generate_particles_no_overlap(
        N=N,
        a=a, b=b, c=c, d=d,
        particle_radius=particle_radius,
        r_min=r_min,
        r_max=r_max,
        safety=safety,
        max_trials=max_trials,
        seed=particle_seed_base + shape_id,
    )

    all_results.append({
        "shape_number": shape_number,
        "a": a,
        "b": b,
        "c": c,
        "d": d,
        "boundary_X": X,
        "boundary_Y": Y,
        "q_particles": q_particles,
        "x_particles": x_particles,
        "nearest": nearest,
        "info": info,
    })

    row = shape_id // ncols
    col = shape_id % ncols

    subgs = outer[row, col].subgridspec(1, 2, wspace=0.18)

    ax_phys = fig.add_subplot(subgs[0, 0])
    ax_q = fig.add_subplot(subgs[0, 1])

    ax_phys.plot(X, Y, linewidth=0.5, color="black")
    ax_phys.scatter(
        x_particles[:, 0],
        x_particles[:, 1],
        s=0.5,
    )
    ax_phys.set_aspect("equal", adjustable="box")
    ax_phys.set_xlabel("x")
    ax_phys.set_ylabel("y")
    ax_phys.grid(True)
    ax_phys.set_title(f"shape {shape_number}\nphysical")

    ax_q.scatter(
        q_particles[:, 1],
        q_particles[:, 0],
        s=1,
        color="red",
    )
    ax_q.set_xlim(0.0, TWO_PI)
    ax_q.set_ylim(0.0, 1.0)
    ax_q.set_xlabel(r"$\theta$")
    ax_q.set_ylabel(r"$r$")
    ax_q.grid(True)
    ax_q.set_title("curvilinear")

fig.savefig(FIGURE_DIR / "case_1_initial_particle_distributions.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
for result in all_results:
    print(f"shape {result['shape_number']}: {result['info']}")

## Curvilinear Simulation With Rotating Gravity


In [ ]:
import importlib
import time

import curvilinear_solver_rotating as csol

csol = importlib.reload(csol)


In [ ]:
# Simulation parameters for the rotating-body-frame study.
dt = 2.0e-4
T_max = 10.0
save_every = 100

particle_mass = 1.0
k_contact = 2.0e4
gamma_contact = 15.0
k_w = 2.0e4
gamma_w = 15.0

g_magnitude = 9.81
omega = 2.0 * np.pi * 0.25

box_padding = 0.5
r_cap = 0.03
r_exit = 0.06
inverse_n = 4096


In [ ]:
def solver_box_from_curve(X, Y, pad=0.5):
    return np.array(
        [X.min() - pad, X.max() + pad, Y.min() - pad, Y.max() + pad],
        dtype=np.float32,
    )


def stack_if_same_shape(arrays):
    shapes = {arr.shape for arr in arrays}
    if len(shapes) == 1:
        return np.stack(arrays, axis=0)
    return arrays


In [ ]:
q0_by_shape = []
p0_by_shape = []
x0_by_shape = []
q_out_by_shape = []
p_out_by_shape = []
x_out_by_shape = []
v_out_by_shape = []
shape_time_by_shape = []
simulation_results = []

for result in all_results:
    shape_number = result["shape_number"]
    a = result["a"]
    b = result["b"]
    c = result["c"]
    d = result["d"]
    boundary_X = result["boundary_X"]
    boundary_Y = result["boundary_Y"]

    q0 = result["q_particles"].astype(np.float32)
    p0 = np.zeros_like(q0, dtype=np.float32)
    x0 = result["x_particles"].astype(np.float32)

    n_particles = q0.shape[0]
    m = np.full(n_particles, particle_mass, dtype=np.float32)
    rad = np.full(n_particles, particle_radius, dtype=np.float32)
    group_mobile = np.array([0, n_particles], dtype=np.int64)
    box = solver_box_from_curve(boundary_X, boundary_Y, pad=box_padding)

    start = time.perf_counter()
    t_out, q_out, p_out, x_out, v_out = csol.simulate_curvilinear_particles(
        box=box,
        q0=q0,
        p0=p0,
        m=m,
        rad=rad,
        dt=dt,
        T_max=T_max,
        group_mobile=group_mobile,
        k_contact=k_contact,
        gamma_contact=gamma_contact,
        k_w=k_w,
        gamma_w=gamma_w,
        g_magnitude=g_magnitude,
        omega=omega,
        a=a,
        b=b,
        c=c,
        d=d,
        save_every=save_every,
        r_cap=r_cap,
        r_exit=r_exit,
        inverse_n=inverse_n,
    )
    runtime = time.perf_counter() - start

    q0_by_shape.append(q0)
    p0_by_shape.append(p0)
    x0_by_shape.append(x0)
    q_out_by_shape.append(q_out)
    p_out_by_shape.append(p_out)
    x_out_by_shape.append(x_out)
    v_out_by_shape.append(v_out)
    shape_time_by_shape.append(t_out)

    result["q0"] = q0
    result["p0"] = p0
    result["x0"] = x0
    result["t_out"] = t_out
    result["q_out"] = q_out
    result["p_out"] = p_out
    result["x_out"] = x_out
    result["v_out"] = v_out
    result["runtime"] = runtime

    simulation_results.append({
        "shape_number": shape_number,
        "a": a,
        "b": b,
        "c": c,
        "d": d,
        "boundary_X": boundary_X,
        "boundary_Y": boundary_Y,
        "box": box,
        "q0": q0,
        "p0": p0,
        "x0": x0,
        "t": t_out,
        "q": q_out,
        "p": p_out,
        "x": x_out,
        "v": v_out,
        "runtime": runtime,
    })

    print(
        f"shape {shape_number}: saved {len(t_out)} frames, "
        f"final t={float(t_out[-1]):.3f}, runtime={runtime:.3f} s"
    )

q0_array = stack_if_same_shape(q0_by_shape)
p0_array = stack_if_same_shape(p0_by_shape)
x0_array = stack_if_same_shape(x0_by_shape)
q_out_array = stack_if_same_shape(q_out_by_shape)
p_out_array = stack_if_same_shape(p_out_by_shape)
x_out_array = stack_if_same_shape(x_out_by_shape)
v_out_array = stack_if_same_shape(v_out_by_shape)
time_array = stack_if_same_shape(shape_time_by_shape)


## Lab-Frame Rotating Visualization


In [ ]:
def rotate_points(points, angle):
    points = np.asarray(points)
    c = np.cos(angle)
    s = np.sin(angle)
    rotated = np.empty_like(points, dtype=float)
    rotated[..., 0] = c * points[..., 0] - s * points[..., 1]
    rotated[..., 1] = s * points[..., 0] + c * points[..., 1]
    return rotated


def lab_rotation_angle(time_value, omega, gravity_angle=-0.5 * np.pi):
    """
    Convert body-frame solver coordinates to the lab view.

    The solver uses g_body(t) = g [cos(omega t), sin(omega t)].
    This rotation maps that vector to a fixed lab-frame gravity direction.
    The default gravity_angle is downward along negative y.
    """
    return gravity_angle - float(omega) * float(time_value)


def boundary_points(sim):
    return np.column_stack((sim["boundary_X"], sim["boundary_Y"]))


def lab_snapshot(sim, frame_id, omega):
    angle = lab_rotation_angle(float(sim["t"][frame_id]), omega)
    wall_lab = rotate_points(boundary_points(sim), angle)
    particles_lab = rotate_points(sim["x"][frame_id], angle)
    return wall_lab, particles_lab


def plot_gravity_arrow(ax):
    ax.annotate(
        "",
        xy=(0.90, 0.08),
        xytext=(0.90, 0.24),
        xycoords="axes fraction",
        arrowprops={"arrowstyle": "->", "color": "black", "lw": 1.0},
    )
    ax.text(0.93, 0.08, "g", transform=ax.transAxes, va="center")


## Final Saved Configuration


In [ ]:
from matplotlib.patches import FancyArrowPatch

def add_rotation_arrow(ax, extent, omega):
    # Positive omega appears clockwise after the lab-frame transform used here.
    if omega >= 0:
        theta1, theta2 = 130, -210
    else:
        theta1, theta2 = -50, 290

    arrow = FancyArrowPatch(
        (-0.16 * extent, 0.05 * extent),
        (0.16 * extent, 0.05 * extent),
        connectionstyle=f"arc3,rad={-0.65 if omega >= 0 else 0.65}",
        arrowstyle="-|>",
        mutation_scale=14,
        lw=1.8,
        color="black",
        zorder=6,
    )
    ax.add_patch(arrow)


snapshot_id = -1
ncols = 3
nrows = int(np.ceil(n_shapes / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(10, 10))
axes = axes.ravel()

for shape_id, sim in enumerate(simulation_results):
    wall_lab, particles_lab = lab_snapshot(sim, snapshot_id, omega)
    extent = 1.10 * np.sqrt(np.sum(boundary_points(sim)**2, axis=1)).max()

    ax = axes[shape_id]
    ax.plot(wall_lab[:, 0], wall_lab[:, 1], color="black", lw=0.8)
    ax.scatter(
        particles_lab[:, 0],
        particles_lab[:, 1],
        s=0.5,
    )
    plot_gravity_arrow(ax)
    add_rotation_arrow(ax, extent, omega)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-extent, extent)
    ax.set_ylim(-extent, extent)
    ax.set_title(f"shape {sim['shape_number']}")
    ax.grid(True)

for ax in axes[len(simulation_results):]:
    ax.axis("off")

plt.tight_layout()
fig.savefig(FIGURE_DIR / "case_1_rotating_final_snapshot.png", dpi=300, bbox_inches="tight")
plt.show()

## Rotating Video


In [ ]:
from matplotlib import animation
from IPython.display import HTML

# Increase the embedded-animation limit for the notebook HTML preview.
plt.rcParams["animation.embed_limit"] = 500  # MB

video_start_time = 2.0
video_end_time = T_max
video_stride = 1
video_fps = 20

t_video = simulation_results[0]["t"]
frame_ids = np.nonzero((t_video >= video_start_time) & (t_video <= video_end_time))[0][::video_stride]
if len(frame_ids) == 0:
    raise ValueError("No frames selected. Check video_start_time, video_end_time, and save_every.")

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.ravel()

wall_lines = []
particle_scatters = []

for shape_id, sim in enumerate(simulation_results):
    extent = 1.10 * np.sqrt(np.sum(boundary_points(sim)**2, axis=1)).max()

    ax = axes[shape_id]
    line, = ax.plot([], [], color="black", lw=0.8)
    scatter = ax.scatter([], [], s=0.5)
    plot_gravity_arrow(ax)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-extent, extent)
    ax.set_ylim(-extent, extent)
    ax.set_title(f"shape {sim['shape_number']}")
    ax.set_xticks([])
    ax.set_yticks([])

    wall_lines.append(line)
    particle_scatters.append(scatter)

for ax in axes[len(simulation_results):]:
    ax.axis("off")

# Keep the time label away from the subplot titles.
time_text = fig.text(0.98, 0.012, "", ha="right", va="bottom", fontsize=11)
fig.subplots_adjust(left=0.03, right=0.98, top=0.95, bottom=0.05, wspace=0.08, hspace=0.18)


def update(frame_id):
    artists = []
    for shape_id, sim in enumerate(simulation_results):
        wall_lab, particles_lab = lab_snapshot(sim, int(frame_id), omega)
        wall_lines[shape_id].set_data(wall_lab[:, 0], wall_lab[:, 1])
        particle_scatters[shape_id].set_offsets(particles_lab)
        artists.extend([wall_lines[shape_id], particle_scatters[shape_id]])

    time_text.set_text(f"t = {float(t_video[int(frame_id)]):.3f}")
    artists.append(time_text)
    return artists

ani = animation.FuncAnimation(
    fig,
    update,
    frames=frame_ids,
    interval=1000.0 / video_fps,
    blit=False,
)

html_path = VIDEO_DIR / "case_1_all_shapes_rotating_t2_to_t5.html"
ani.save(
    html_path,
    writer=animation.HTMLWriter(fps=video_fps, embed_frames=True, default_mode="loop"),
)

avi_path = VIDEO_DIR / "case_1_all_shapes_rotating_t2_to_t5.avi"
if animation.writers.is_available("ffmpeg"):
    ani.save(
        avi_path,
        writer=animation.FFMpegWriter(fps=video_fps, codec="mpeg4", bitrate=3000),
        dpi=150,
    )
    print(f"saved AVI: {avi_path}")
else:
    print("ffmpeg is not available, so the AVI file was not written. The HTML animation was saved.")

plt.close(fig)
print(f"saved HTML: {html_path}")

HTML(html_path.read_text(encoding="utf-8"))


In [ ]:
## Curvilinear-Coordinate View With Rotating Boundary

In [ ]:
def make_inverse_table_for_shape(a, b, c, d, n=4096):
    theta_table = np.linspace(0.0, TWO_PI, int(n) + 1)
    Bx, By = fourier_curve(theta_table, a, b, c, d)
    phi_table = np.unwrap(np.arctan2(By, Bx))

    if phi_table[-1] < phi_table[0]:
        phi_table = phi_table[::-1].copy()
        theta_table = theta_table[::-1].copy()

    return theta_table, phi_table


def physical_points_to_q(points, a, b, c, d, theta_table, phi_table):
    x = points[:, 0]
    y = points[:, 1]

    phi = np.arctan2(y, x)
    phi0 = phi_table[0]
    phi1 = phi_table[-1]

    phi_adj = phi.copy()
    phi_adj += TWO_PI * np.ceil((phi0 - phi_adj) / TWO_PI)
    phi_adj -= TWO_PI * np.floor((phi_adj - phi0) / TWO_PI)
    phi_adj = np.clip(phi_adj, phi0, phi1)

    theta = np.interp(phi_adj, phi_table, theta_table)
    theta = np.mod(theta, TWO_PI)

    Bx, By = fourier_curve(theta, a, b, c, d)
    den = Bx * Bx + By * By
    r = (x * Bx + y * By) / den

    return theta, r


def sorted_q_curve(theta, r):
    order = np.argsort(theta)
    return theta[order], r[order]


inverse_tables = []
for sim in simulation_results:
    inverse_tables.append(
        make_inverse_table_for_shape(
            sim["a"], sim["b"], sim["c"], sim["d"], n=inverse_n
        )
    )


def lab_to_q_snapshot(sim, frame_id, omega, inverse_table):
    angle = lab_rotation_angle(float(sim["t"][frame_id]), omega)

    wall_lab = rotate_points(boundary_points(sim), angle)
    particles_lab = rotate_points(sim["x"][frame_id], angle)

    theta_table, phi_table = inverse_table

    theta_wall, r_wall = physical_points_to_q(
        wall_lab, sim["a"], sim["b"], sim["c"], sim["d"], theta_table, phi_table
    )
    theta_particles, r_particles = physical_points_to_q(
        particles_lab, sim["a"], sim["b"], sim["c"], sim["d"], theta_table, phi_table
    )

    return theta_wall, r_wall, theta_particles, r_particles

In [ ]:
snapshot_id = -1
ncols = 3
nrows = int(np.ceil(n_shapes / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(10, 10))
axes = axes.ravel()

for shape_id, sim in enumerate(simulation_results):
    theta_wall, r_wall, theta_particles, r_particles = lab_to_q_snapshot(
        sim, snapshot_id, omega, inverse_tables[shape_id]
    )

    theta_wall_plot, r_wall_plot = sorted_q_curve(theta_wall, r_wall)

    ax = axes[shape_id]
    ax.plot(theta_wall_plot, r_wall_plot, color="black", lw=0.8)
    ax.scatter(theta_particles, r_particles, s=0.5)

    ax.set_xlim(0.0, TWO_PI)
    ax.set_ylim(0.0, 2.00)
    ax.set_xlabel(r"$\theta$")
    ax.set_ylabel(r"$r$")
    ax.set_title(f"shape {sim['shape_number']}")
    ax.grid(True)

for ax in axes[len(simulation_results):]:
    ax.axis("off")

plt.tight_layout()
fig.savefig(
    FIGURE_DIR / "case_1_curvilinear_rotating_boundary_snapshot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
from matplotlib import animation
from IPython.display import HTML

plt.rcParams["animation.embed_limit"] = 500  # MB

video_start_time = 2.0
video_end_time = T_max
video_stride = 1
video_fps = 20

t_video = simulation_results[0]["t"]
frame_ids = np.nonzero((t_video >= video_start_time) & (t_video <= video_end_time))[0][::video_stride]

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.ravel()

wall_lines = []
particle_scatters = []

for shape_id, sim in enumerate(simulation_results):
    ax = axes[shape_id]
    line, = ax.plot([], [], color="black", lw=0.8)
    scatter = ax.scatter([], [], s=0.5)

    ax.set_xlim(0.0, TWO_PI)
    ax.set_ylim(0.0, 2.00)
    ax.set_title(f"shape {sim['shape_number']}")
    ax.set_xticks([])
    ax.set_yticks([])

    wall_lines.append(line)
    particle_scatters.append(scatter)

time_text = fig.text(0.98, 0.012, "", ha="right", va="bottom", fontsize=11)
fig.subplots_adjust(left=0.03, right=0.98, top=0.95, bottom=0.05, wspace=0.08, hspace=0.18)


def update_q_video(frame_id):
    artists = []

    for shape_id, sim in enumerate(simulation_results):
        theta_wall, r_wall, theta_particles, r_particles = lab_to_q_snapshot(
            sim, int(frame_id), omega, inverse_tables[shape_id]
        )

        theta_wall_plot, r_wall_plot = sorted_q_curve(theta_wall, r_wall)

        wall_lines[shape_id].set_data(theta_wall_plot, r_wall_plot)
        particle_scatters[shape_id].set_offsets(
            np.column_stack((theta_particles, r_particles))
        )

        artists.extend([wall_lines[shape_id], particle_scatters[shape_id]])

    time_text.set_text(f"t = {float(t_video[int(frame_id)]):.3f}")
    artists.append(time_text)
    return artists


ani_q = animation.FuncAnimation(
    fig,
    update_q_video,
    frames=frame_ids,
    interval=1000.0 / video_fps,
    blit=False,
)

html_path = VIDEO_DIR / "case_1_curvilinear_rotating_boundary_t2_to_t5.html"
ani_q.save(
    html_path,
    writer=animation.HTMLWriter(fps=video_fps, embed_frames=True, default_mode="loop"),
)

avi_path = VIDEO_DIR / "case_1_curvilinear_rotating_boundary_t2_to_t5.avi"
if animation.writers.is_available("ffmpeg"):
    ani_q.save(
        avi_path,
        writer=animation.FFMpegWriter(fps=video_fps, codec="mpeg4", bitrate=3000),
        dpi=150,
    )
    print(f"saved AVI: {avi_path}")
else:
    print("ffmpeg is not available, so the AVI file was not written. The HTML animation was saved.")

plt.close(fig)
print(f"saved HTML: {html_path}")

HTML(html_path.read_text(encoding="utf-8"))

In [ ]:
## Curvilinear-Coordinate View With non Rotating Boundary

In [ ]:
snapshot_id = -1
ncols = 3
nrows = int(np.ceil(n_shapes / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(10, 10))
axes = axes.ravel()

theta_wall = np.linspace(0.0, TWO_PI, 500)
r_wall = np.ones_like(theta_wall)

for shape_id, sim in enumerate(simulation_results):
    q_snapshot = sim["q"][snapshot_id]

    ax = axes[shape_id]
    ax.plot(theta_wall, r_wall, color="black", lw=0.8)
    ax.scatter(
        q_snapshot[:, 1],
        q_snapshot[:, 0],
        s=0.5,
    )

    ax.set_xlim(0.0, TWO_PI)
    ax.set_ylim(0.0, 1.25)
    ax.set_xlabel(r"$\theta$")
    ax.set_ylabel(r"$r$")
    ax.set_title(f"shape {sim['shape_number']}")
    ax.grid(True)

for ax in axes[len(simulation_results):]:
    ax.axis("off")

plt.tight_layout()
fig.savefig(
    FIGURE_DIR / "case_1_curvilinear_fixed_boundary_snapshot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
from matplotlib import animation
from IPython.display import HTML

plt.rcParams["animation.embed_limit"] = 500  # MB

video_start_time = 2.0
video_end_time = T_max
video_stride = 1
video_fps = 20

t_video = simulation_results[0]["t"]
frame_ids = np.nonzero((t_video >= video_start_time) & (t_video <= video_end_time))[0][::video_stride]

theta_wall = np.linspace(0.0, TWO_PI, 500)
r_wall = np.ones_like(theta_wall)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
axes = axes.ravel()

wall_lines = []
particle_scatters = []

for shape_id, sim in enumerate(simulation_results):
    ax = axes[shape_id]
    line, = ax.plot(theta_wall, r_wall, color="black", lw=0.8)
    scatter = ax.scatter([], [], s=0.5)

    ax.set_xlim(0.0, TWO_PI)
    ax.set_ylim(0.0, 1.25)
    ax.set_title(f"shape {sim['shape_number']}")
    ax.set_xticks([])
    ax.set_yticks([])

    wall_lines.append(line)
    particle_scatters.append(scatter)

time_text = fig.text(0.98, 0.012, "", ha="right", va="bottom", fontsize=11)
fig.subplots_adjust(left=0.03, right=0.98, top=0.95, bottom=0.05, wspace=0.08, hspace=0.18)


def update_fixed_q_video(frame_id):
    artists = []

    for shape_id, sim in enumerate(simulation_results):
        q_frame = sim["q"][int(frame_id)]

        particle_scatters[shape_id].set_offsets(
            np.column_stack((q_frame[:, 1], q_frame[:, 0]))
        )

        artists.append(particle_scatters[shape_id])

    time_text.set_text(f"t = {float(t_video[int(frame_id)]):.3f}")
    artists.append(time_text)
    return artists


ani_fixed_q = animation.FuncAnimation(
    fig,
    update_fixed_q_video,
    frames=frame_ids,
    interval=1000.0 / video_fps,
    blit=False,
)

html_path = VIDEO_DIR / "case_1_curvilinear_fixed_boundary_t2_to_t5.html"
ani_fixed_q.save(
    html_path,
    writer=animation.HTMLWriter(fps=video_fps, embed_frames=True, default_mode="loop"),
)

avi_path = VIDEO_DIR / "case_1_curvilinear_fixed_boundary_t2_to_t5.avi"
if animation.writers.is_available("ffmpeg"):
    ani_fixed_q.save(
        avi_path,
        writer=animation.FFMpegWriter(fps=video_fps, codec="mpeg4", bitrate=3000),
        dpi=150,
    )
    print(f"saved AVI: {avi_path}")
else:
    print("ffmpeg is not available, so the AVI file was not written. The HTML animation was saved.")

plt.close(fig)
print(f"saved HTML: {html_path}")

HTML(html_path.read_text(encoding="utf-8"))

In [ ]:
for sim in simulation_results:
    rmax = np.max(sim["q"][:, :, 0])
    print(sim["shape_number"], rmax)